In [13]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import optuna
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import confusion_matrix, roc_auc_score
import numpy as np
from imblearn.under_sampling import RandomUnderSampler

class MyDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

class FlexibleNet(nn.Module):
    def __init__(self, input_dim, hidden_dims, dropout_rate):
        super().__init__()
        layers = []
        last_dim = input_dim
        for h_dim in hidden_dims:
            layers.append(nn.Linear(last_dim, h_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))
            last_dim = h_dim
        layers.append(nn.Linear(last_dim, 1))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)

def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device).unsqueeze(1)
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * X_batch.size(0)
    return running_loss / len(loader.dataset)

def evaluate(model, loader, criterion, device):
    model.eval()
    all_preds = []
    all_targets = []
    running_loss = 0.0
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device).unsqueeze(1)
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            running_loss += loss.item() * X_batch.size(0)
            preds = torch.sigmoid(outputs)
            all_preds.append(preds.cpu())
            all_targets.append(y_batch.cpu())
    avg_loss = running_loss / len(loader.dataset)
    preds = torch.cat(all_preds).numpy()
    targets = torch.cat(all_targets).numpy()
    return avg_loss, preds, targets

def train_and_tune(train_df, test_df, n_trials=30, epochs=20):
    train_df = train_df.drop(columns=['wavelet'])
    test_df = test_df.drop(columns=['wavelet'])
    X_train = train_df.drop(columns=['label']).values
    y_train = train_df['label'].values
    X_test = test_df.drop(columns=['label']).values
    y_test = test_df['label'].values
    
    scaler = MinMaxScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # SMOTE to balance training data
    rus = RandomUnderSampler(sampling_strategy = 0.4,random_state=42)
    X_train_bal, y_train_bal = rus.fit_resample(X_train, y_train)

    # Compute pos_weight for BCEWithLogitsLoss
    classes, counts = np.unique(y_train_bal, return_counts=True)
    class_weights = {cls: len(y_train_bal) / count for cls, count in zip(classes, counts)}
    pos_weight = torch.tensor([class_weights.get(1, 1.0) / class_weights.get(0, 1.0)], dtype=torch.float32).to(device)
    
    def objective(trial):
        n_layers = trial.suggest_int("n_layers", 3, 5)
        hidden_dims = []
        for i in range(n_layers):
            hidden_dims.append(trial.suggest_int(f"n_units_l{i}", 16, 100))
        dropout_rate = trial.suggest_float("dropout_rate", 0.0, 0.5)
        lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
        batch_size = trial.suggest_categorical("batch_size", [16, 32, 64, 128])
        
        idxs = np.arange(len(X_train_bal))
        np.random.shuffle(idxs)
        split = int(len(idxs)*0.8)
        train_idx, val_idx = idxs[:split], idxs[split:]
        
        train_dataset = MyDataset(X_train_bal[train_idx], y_train_bal[train_idx])
        val_dataset = MyDataset(X_train_bal[val_idx], y_train_bal[val_idx])
        
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=batch_size)
        
        model = FlexibleNet(input_dim=X_train_bal.shape[1], hidden_dims=hidden_dims, dropout_rate=dropout_rate).to(device)
        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        optimizer = optim.Adam(model.parameters(), lr=lr)
        
        for epoch in range(epochs):
            train_epoch(model, train_loader, criterion, optimizer, device)
            val_loss, val_preds_prob, val_targets = evaluate(model, val_loader, criterion, device)
            val_auc = roc_auc_score(val_targets, val_preds_prob)
            trial.report(val_auc, epoch)
            if trial.should_prune():
                raise optuna.exceptions.TrialPruned()
        return val_auc
    
    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials)
    
    print(f"Best trial AUROC: {study.best_value:.4f}")
    print("Best params:", study.best_params)
    
    # Final training on full balanced training data
    best_params = study.best_params
    n_layers = best_params["n_layers"]
    hidden_dims = [best_params[f"n_units_l{i}"] for i in range(n_layers)]
    batch_size = best_params["batch_size"]
    dropout_rate = best_params["dropout_rate"]
    lr = best_params["lr"]
    
    final_dataset = MyDataset(X_train_bal, y_train_bal)
    final_loader = DataLoader(final_dataset, batch_size=batch_size, shuffle=True)
    
    final_model = FlexibleNet(input_dim=X_train_bal.shape[1], hidden_dims=hidden_dims, dropout_rate=dropout_rate).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = optim.Adam(final_model.parameters(), lr=lr)
    
    for epoch in range(epochs):
        train_epoch(final_model, final_loader, criterion, optimizer, device)
    
    # Evaluate on original test set
    test_dataset = MyDataset(X_test, y_test)
    test_loader = DataLoader(test_dataset, batch_size=batch_size)
    
    _, test_preds_prob, test_targets = evaluate(final_model, test_loader, criterion, device)
    test_preds = (test_preds_prob > 0.5).astype(int)
    cm = confusion_matrix(test_targets, test_preds)
    
    return final_model, cm, study.best_trial


In [23]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import optuna
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import confusion_matrix, roc_auc_score
import numpy as np
from imblearn.under_sampling import RandomUnderSampler
from sklearn.model_selection import StratifiedKFold
from torch.optim.lr_scheduler import StepLR, ReduceLROnPlateau

class MyDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

def get_activation(name):
    if name == "relu":
        return nn.ReLU()
    elif name == "leakyrelu":
        return nn.LeakyReLU()
    elif name == "elu":
        return nn.ELU()
    else:
        raise ValueError("Unknown activation")

def get_optimizer(name, params, lr, weight_decay):
    if name == "adam":
        return optim.Adam(params, lr=lr, weight_decay=weight_decay)
    elif name == "adamw":
        return optim.AdamW(params, lr=lr, weight_decay=weight_decay)
    elif name == "sgd":
        return optim.SGD(params, lr=lr, weight_decay=weight_decay, momentum=0.9)
    else:
        raise ValueError("Unknown optimizer")

def build_model(input_dim, hidden_dims, dropout_rate, activation_name, batch_norm):
    layers = []
    last_dim = input_dim
    for h_dim in hidden_dims:
        layers.append(nn.Linear(last_dim, h_dim))
        if batch_norm:
            layers.append(nn.BatchNorm1d(h_dim))
        layers.append(get_activation(activation_name))
        layers.append(nn.Dropout(dropout_rate))
        last_dim = h_dim
    layers.append(nn.Linear(last_dim, 1))
    return nn.Sequential(*layers)

def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device).unsqueeze(1)
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * X_batch.size(0)
    return running_loss / len(loader.dataset)

def evaluate(model, loader, criterion, device):
    model.eval()
    all_preds = []
    all_targets = []
    running_loss = 0.0
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device).unsqueeze(1)
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            running_loss += loss.item() * X_batch.size(0)
            preds = torch.sigmoid(outputs)
            all_preds.append(preds.cpu())
            all_targets.append(y_batch.cpu())
    avg_loss = running_loss / len(loader.dataset)
    preds = torch.cat(all_preds).numpy()
    targets = torch.cat(all_targets).numpy()
    return avg_loss, preds, targets

def train_and_tune_advanced(train_df, test_df, n_trials=50, k_folds=3):
    train_df = train_df.drop(columns=['wavelet'])
    test_df = test_df.drop(columns=['wavelet'])
    X_train = train_df.drop(columns=['label']).values
    y_train = train_df['label'].values
    X_test = test_df.drop(columns=['label']).values
    y_test = test_df['label'].values
    
    scaler = MinMaxScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Undersample 
    rus = RandomUnderSampler(sampling_strategy=0.4, random_state=42)
    X_train_bal, y_train_bal = rus.fit_resample(X_train, y_train)

    def objective(trial):
        n_layers = trial.suggest_int("n_layers", 2, 8)
        hidden_dims = [trial.suggest_int(f"n_units_l{i}", 16, 512) for i in range(n_layers)]
        dropout_rate = trial.suggest_float("dropout_rate", 0.0, 0.7)
        lr = trial.suggest_float("lr", 1e-5, 1e-1, log=True)
        weight_decay = trial.suggest_float("weight_decay", 1e-8, 1e-2, log=True)
        optimizer_name = trial.suggest_categorical("optimizer", ["adam", "adamw", "sgd"])
        activation_name = trial.suggest_categorical("activation", ["relu", "leakyrelu", "elu"])
        batch_norm = trial.suggest_categorical("batch_norm", [True, False])
        scheduler_name = trial.suggest_categorical("scheduler", ["none", "steplr", "reduceonplateau"])
        batch_size = trial.suggest_categorical("batch_size", [16, 32, 64, 128])
        max_epochs = trial.suggest_int("epochs", 10, 50)

        pos_weight = torch.tensor([sum(y_train_bal==0)/sum(y_train_bal==1)], dtype=torch.float32).to(device)
        skf = StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=42)
        val_aucs = []
        for train_idx, val_idx in skf.split(X_train_bal, y_train_bal):
            model = build_model(X_train_bal.shape[1], hidden_dims, dropout_rate, activation_name, batch_norm).to(device)
            optimizer = get_optimizer(optimizer_name, model.parameters(), lr, weight_decay)
            if scheduler_name == "steplr":
                scheduler = StepLR(optimizer, step_size=10, gamma=0.5)
            elif scheduler_name == "reduceonplateau":
                scheduler = ReduceLROnPlateau(optimizer, mode='max', patience=3)
            else:
                scheduler = None
            criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
            train_dataset = MyDataset(X_train_bal[train_idx], y_train_bal[train_idx])
            val_dataset = MyDataset(X_train_bal[val_idx], y_train_bal[val_idx])
            train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
            val_loader = DataLoader(val_dataset, batch_size=batch_size)
            for epoch in range(max_epochs):
                train_epoch(model, train_loader, criterion, optimizer, device)
                _, val_preds_prob, val_targets = evaluate(model, val_loader, criterion, device)
                val_auc = roc_auc_score(val_targets, val_preds_prob)
                if scheduler_name == "reduceonplateau":
                    scheduler.step(val_auc)
                elif scheduler_name == "steplr":
                    scheduler.step()
                trial.report(val_auc, epoch)
                if trial.should_prune():
                    raise optuna.exceptions.TrialPruned()
            val_aucs.append(val_auc)
        return np.mean(val_aucs)

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials)
    print(f"Best trial AUROC: {study.best_value:.4f}")
    print("Best params:", study.best_params)

    # Final training on full balanced training data with best params
    best_params = study.best_params
    n_layers = best_params["n_layers"]
    hidden_dims = [best_params[f"n_units_l{i}"] for i in range(n_layers)]
    batch_size = best_params["batch_size"]
    dropout_rate = best_params["dropout_rate"]
    lr = best_params["lr"]
    weight_decay = best_params["weight_decay"]
    optimizer_name = best_params["optimizer"]
    activation_name = best_params["activation"]
    batch_norm = best_params["batch_norm"]
    scheduler_name = best_params["scheduler"]
    max_epochs = best_params["epochs"]

    pos_weight = torch.tensor([sum(y_train_bal==0)/sum(y_train_bal==1)], dtype=torch.float32).to(device)
    final_model = build_model(X_train_bal.shape[1], hidden_dims, dropout_rate, activation_name, batch_norm).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = get_optimizer(optimizer_name, final_model.parameters(), lr, weight_decay)
    if scheduler_name == "steplr":
        scheduler = StepLR(optimizer, step_size=10, gamma=0.5)
    elif scheduler_name == "reduceonplateau":
        scheduler = ReduceLROnPlateau(optimizer, mode='max', patience=3)
    else:
        scheduler = None
    final_dataset = MyDataset(X_train_bal, y_train_bal)
    final_loader = DataLoader(final_dataset, batch_size=batch_size, shuffle=True)
    for epoch in range(max_epochs):
        train_epoch(final_model, final_loader, criterion, optimizer, device)
        if scheduler_name == "reduceonplateau":
            _, preds, targets = evaluate(final_model, final_loader, criterion, device)
            auc = roc_auc_score(targets, preds)
            scheduler.step(auc)
        elif scheduler_name == "steplr":
            scheduler.step()

    # Evaluate on original test set
    test_dataset = MyDataset(X_test, y_test)
    test_loader = DataLoader(test_dataset, batch_size=batch_size)
    _, test_preds_prob, test_targets = evaluate(final_model, test_loader, criterion, device)
    test_preds = (test_preds_prob > 0.5).astype(int)
    cm = confusion_matrix(test_targets, test_preds)
    return final_model, cm, study.best_trial


In [24]:
import pandas as pd

In [25]:
# tr1 = pd.read_csv(r"C:\bt23ece064\wavelet\saved_dfs\train_df_1.csv")
# ts1 = pd.read_csv(r"C:\bt23ece064\wavelet\saved_dfs\test_df_1.csv")

# final_model,cm,best_trial = train_and_tune_advanced(tr1,ts1)

In [26]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
conf_matrices = []
for i in range(106):
    train = pd.read_csv(f"C:\\bt23ece064\\wavelet\\saved_dfs\\train_df_{i+1}.csv")
    test = pd.read_csv(f"C:\\bt23ece064\\wavelet\\saved_dfs\\test_df_{i+1}.csv")
    model, cm, study = train_and_tune_advanced(train, test)
    conf_matrices.append(cm)
    print(f"Confusion matrix for fold {i+1}:")
    np.savetxt(f'C:\\bt23ece064\\wavelet\\confusion_matrix{i+1}.txt', cm, delimiter=',')
    print(cm)

[I 2025-07-01 17:02:37,339] A new study created in memory with name: no-name-e206273d-9023-4e86-ae4c-3ac58ec5bed9
[I 2025-07-01 17:03:35,551] Trial 0 finished with value: 0.5539428766361764 and parameters: {'n_layers': 6, 'n_units_l0': 428, 'n_units_l1': 198, 'n_units_l2': 428, 'n_units_l3': 424, 'n_units_l4': 109, 'n_units_l5': 24, 'dropout_rate': 0.5981749418209316, 'lr': 0.0004958624293417364, 'weight_decay': 0.00449091652993129, 'optimizer': 'adam', 'activation': 'leakyrelu', 'batch_norm': True, 'scheduler': 'none', 'batch_size': 16, 'epochs': 22}. Best is trial 0 with value: 0.5539428766361764.
[I 2025-07-01 17:04:06,547] Trial 1 finished with value: 0.5392124875437911 and parameters: {'n_layers': 2, 'n_units_l0': 86, 'n_units_l1': 347, 'dropout_rate': 0.5945296169061676, 'lr': 0.013195235517121042, 'weight_decay': 0.001529186991569234, 'optimizer': 'sgd', 'activation': 'leakyrelu', 'batch_norm': True, 'scheduler': 'steplr', 'batch_size': 16, 'epochs': 43}. Best is trial 0 with va